# Xem dataset phòng thi (ảnh hoặc video) trước khi train

Notebook không train gì. Nó chỉ in cấu trúc thư mục, tên lớp, số nhãn mỗi lớp và giấy phép. Với ảnh nó lưu `mau.jpg` (16 ảnh ngẫu nhiên); với video nó in độ phân giải, fps, độ dài, và lưu `mau_video.jpg` (khung giữa của tối đa 16 video). Dựa vào đó Claude quyết định có dùng dataset để train hay không.

**Làm lần lượt:** Add Input dataset (ví dụ `fairouzfarouq/student-cheating-behavior-dataset` hoặc `cctvdataset/cctv-exam-monitor-dataset`), bấm **Run All** (CPU là đủ, không cần GPU), rồi chụp màn hình phần in ra cùng `mau.jpg` / `mau_video.jpg` gửi lại.


In [ ]:
import random
from collections import Counter
from pathlib import Path

DATA_ROOT = Path("/kaggle/input")  # đổi nếu chỉ muốn xem một dataset
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
SKIP_EXTS = {".mp4", ".avi", ".mov", ".mkv"}

files = [p for p in DATA_ROOT.rglob("*") if p.is_file() and p.suffix.lower() not in SKIP_EXTS]
print(f"{len(files)} file dưới {DATA_ROOT}")
print("Theo đuôi:", dict(Counter(p.suffix.lower() for p in files).most_common()))

def tree(root, depth=3, prefix=""):
    if depth == 0:
        return
    kids = sorted(p for p in root.iterdir() if p.is_dir())
    for kid in kids[:30]:
        count = sum(1 for _ in kid.rglob("*") if _.is_file())
        print(f"{prefix}{kid.name}/  ({count} file)")
        tree(kid, depth - 1, prefix + "    ")
    if len(kids) > 30:
        print(f"{prefix}... và {len(kids) - 30} thư mục nữa")

tree(DATA_ROOT)


In [ ]:
# Tên lớp (data.yaml kiểu YOLO), số hộp mỗi lớp, hoặc số ảnh mỗi thư mục lớp
names = {}
for yml in [p for p in files if p.suffix.lower() in (".yaml", ".yml")]:
    text = yml.read_text(encoding="utf-8", errors="replace")
    print(f"--- {yml} ---")
    print(text[:2000])
    try:
        import yaml
        data = yaml.safe_load(text) or {}
        raw = data.get("names")
        if isinstance(raw, list):
            names = dict(enumerate(raw))
        elif isinstance(raw, dict):
            names = {int(k): v for k, v in raw.items()}
    except Exception as exc:  # noqa: BLE001
        print("Không đọc được yaml:", exc)

label_files = [p for p in files if p.suffix.lower() == ".txt" and "labels" in p.parts]
boxes = Counter()
for path in label_files:
    for line in path.read_text(errors="replace").splitlines():
        parts = line.split()
        if len(parts) >= 5 and parts[0].isdigit():
            boxes[int(parts[0])] += 1
if boxes:
    print(f"\nNhãn YOLO: {len(label_files)} file, {sum(boxes.values())} hộp")
    for cls, count in sorted(boxes.items()):
        print(f"  lớp {cls} ({names.get(cls, '?')}): {count} hộp")
else:
    images = [p for p in files if p.suffix.lower() in IMAGE_EXTS]
    per_folder = Counter(p.parent.name for p in images)
    print(f"\nKhông có nhãn YOLO. {len(images)} ảnh; số ảnh theo thư mục cha (có thể là tên lớp):")
    for folder, count in per_folder.most_common(30):
        print(f"  {folder}: {count}")
for other in [p for p in files if p.suffix.lower() in (".csv", ".json") and p.stat().st_size < 5_000_000][:5]:
    print(f"\n--- {other} (đầu file) ---")
    print(other.read_text(encoding="utf-8", errors="replace")[:800])


In [ ]:
# Giấy phép và mô tả (dataset-metadata.json, README, LICENSE)
for p in files:
    if p.name.lower() in ("dataset-metadata.json", "readme.md", "readme.txt", "license", "license.txt", "license.md"):
        print(f"--- {p} ---")
        print(p.read_text(encoding="utf-8", errors="replace")[:3000])
print("Nếu không thấy gì ở trên: mở trang dataset trên Kaggle, mục License ở cột phải, rồi chụp lại.")


In [ ]:
# Ảnh mẫu: 16 ảnh ngẫu nhiên ghép 4x4, lưu /kaggle/working/mau.jpg
import cv2
import numpy as np

images = [p for p in files if p.suffix.lower() in IMAGE_EXTS]
random.Random(0).shuffle(images)
tiles = []
for p in images[:16]:
    img = cv2.imread(str(p))
    if img is None:
        continue
    img = cv2.resize(img, (320, 240))
    cv2.putText(img, p.parent.name[:30], (4, 16), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 255, 255), 1)
    tiles.append(img)
while tiles and len(tiles) < 16:
    tiles.append(np.zeros_like(tiles[0]))
if tiles:
    sheet = np.vstack([np.hstack(tiles[i:i + 4]) for i in range(0, 16, 4)])
    out = Path("/kaggle/working/mau.jpg") if Path("/kaggle/working").is_dir() else Path("mau.jpg")
    cv2.imwrite(str(out), sheet)
    print("Đã lưu", out)
else:
    print("Không có ảnh nào")


In [ ]:
# Video: số video theo thư mục (có thể là tên lớp), độ phân giải / fps / độ dài, khung giữa ghép 4x4
import cv2
import numpy as np

videos = sorted(p for p in DATA_ROOT.rglob("*") if p.is_file() and p.suffix.lower() in SKIP_EXTS)
print(f"{len(videos)} video")
for folder, count in Counter(p.parent.name for p in videos).most_common(30):
    print(f"  {folder}: {count}")
specs = Counter()
total_s = 0.0
tiles = []
for p in videos:
    cap = cv2.VideoCapture(str(p))
    w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps, n = cap.get(cv2.CAP_PROP_FPS) or 0.0, int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    seconds = n / fps if fps else 0.0
    total_s += seconds
    specs[f"{w}x{h} @ {fps:.0f} fps"] += 1
    print(f"  {p.relative_to(DATA_ROOT)}: {w}x{h}, {fps:.1f} fps, {seconds:.1f} s")
    if len(tiles) < 16 and n > 0:
        cap.set(cv2.CAP_PROP_POS_FRAMES, n // 2)
        ok, img = cap.read()
        if ok:
            img = cv2.resize(img, (320, 180))
            cv2.putText(img, f"{p.parent.name[:18]}/{p.stem[:12]}", (4, 16), cv2.FONT_HERSHEY_SIMPLEX, 0.42,
                        (0, 255, 255), 1)
            tiles.append(img)
    cap.release()
if videos:
    print("Cỡ khung:", dict(specs.most_common()), f"| tổng {total_s / 60:.1f} phút")
while tiles and len(tiles) % 4:
    tiles.append(np.zeros_like(tiles[0]))
if tiles:
    sheet = np.vstack([np.hstack(tiles[i:i + 4]) for i in range(0, len(tiles), 4)])
    out = Path("/kaggle/working/mau_video.jpg") if Path("/kaggle/working").is_dir() else Path("mau_video.jpg")
    cv2.imwrite(str(out), sheet)
    print("Đã lưu", out)
